# EvoPrompt iter2: paired bootstrap значимость before/after

Этот ноутбук делает **paired-анализ по пользователям** для `model × cluster × metric`:

- `answer_similarity`
- `mae_35`
- `trait_similarity`
- `facet_similarity`

Логика:

1. Рекурсивно ищет before/after файлы (`csv/json/jsonl`) в `RESULTS_ROOT`.
2. Выбирает реальные per-user файлы (без хардкода одного имени).
3. Собирает paired users (`after_i - before_i` для одного `user_id`).
4. Считает bootstrap CI для mean delta:
   - `n_resamples=10_000`
   - `confidence_level=0.95`
   - seed `42`
   - `scipy.stats.bootstrap(..., method='BCa')`, при падении fallback на percentile.
5. Строит summary/pivot и сохраняет результаты в `stat_significance/`.


In [19]:
from __future__ import annotations

import importlib.util
import json
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import bootstrap

try:
    from tqdm.auto import tqdm
except Exception:  # noqa: BLE001
    def tqdm(x, *args, **kwargs):
        return x


In [20]:
# --- Path config ---
REPO_ROOT = Path('../').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

# Прямой путь к результатам EvoPrompt iter2 (папки моделей внутри)
RESULTS_ROOT = REPO_ROOT / 'results_experiments' / 'evoprompt_iter2'
if not RESULTS_ROOT.exists():
    raise FileNotFoundError(f'RESULTS_ROOT не найден: {RESULTS_ROOT}')

OUTPUT_DIR = RESULTS_ROOT / 'stat_significance'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RESULTS_ROOT:', RESULTS_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

model_dirs = [
    p.name
    for p in sorted(RESULTS_ROOT.iterdir())
    if p.is_dir() and p.name.lower() not in {'result_log', 'table', 'stat_significance'}
]
print('\nНайденные model-папки:')
for m in model_dirs:
    print('-', m)


REPO_ROOT: D:\programming\GitHub\LLM-PersonaBench
RESULTS_ROOT: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2
OUTPUT_DIR: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2\stat_significance

Найденные model-папки:
- gigchat3
- gpt4_mini
- gpt4_nano
- grok4.1_fast
- qwen3


In [21]:
# --- Try to reuse existing metric code from personality_match.py ---
try:
    from src.utils.personality_match import compute_five_factor_metrics as compute_five_factor_metrics_pm
    PERSONALITY_MATCH_AVAILABLE = True
    PERSONALITY_MATCH_IMPORT_ERROR = None
except Exception as exc:  # noqa: BLE001
    compute_five_factor_metrics_pm = None
    PERSONALITY_MATCH_AVAILABLE = False
    PERSONALITY_MATCH_IMPORT_ERROR = repr(exc)

print('PERSONALITY_MATCH_AVAILABLE:', PERSONALITY_MATCH_AVAILABLE)
if not PERSONALITY_MATCH_AVAILABLE:
    print('Import warning:', PERSONALITY_MATCH_IMPORT_ERROR)
    print(
        'В этой среде недоступны зависимости personality_match.py; '        'но для текущих EvoPrompt iter2 файлов метрики уже есть per-user в participants.jsonl.'
    )


PERSONALITY_MATCH_AVAILABLE: True


In [22]:
# --- Constants ---
USER_ID_CANDIDATES = ['user_id', 'participant_id', 'respondent_id', 'case', 'id']

METRIC_COLUMN_CANDIDATES = {
    'answer_similarity': ['answer_similarity', 'similarity'],
    'mae_35': ['mae_35'],
    'trait_similarity': ['trait_similarity', 'mean_similarity_traits', 'similarity_traits'],
    'facet_similarity': ['facet_similarity', 'mean_similarity_facets', 'similarity_facets'],
}

METRICS = list(METRIC_COLUMN_CANDIDATES.keys())

METRIC_DIRECTIONS = {
    'answer_similarity': 'higher_is_better',
    'mae_35': 'lower_is_better',
    'trait_similarity': 'higher_is_better',
    'facet_similarity': 'higher_is_better',
}


In [23]:
# --- Helpers: parsing and loading ---
def parse_cluster_from_segment(segment: str) -> str | None:
    segment_low = segment.lower()
    tokens = [t for t in re.split(r'[^a-z0-9]+', segment_low) if t]

    for i, tok in enumerate(tokens):
        if tok == 'cluster':
            cluster_ids: list[str] = []
            for nxt in tokens[i + 1:]:
                # Ограничение <=2 цифр помогает не захватывать таймстемпы
                if nxt.isdigit() and len(nxt) <= 2:
                    cluster_ids.append(str(int(nxt)))
                else:
                    break
            if cluster_ids:
                return 'cluster_' + '_'.join(cluster_ids)

    m_c = re.search(r'\bc[_\- ]?(\d{1,2})\b', segment_low)
    if m_c:
        return f"cluster_{int(m_c.group(1))}"

    m_cluster = re.search(r'cluster[_\- ]?(\d{1,2})\b', segment_low)
    if m_cluster:
        return f"cluster_{int(m_cluster.group(1))}"

    return None


def detect_cluster(path: Path) -> str | None:
    for part in reversed(path.parts):
        cluster = parse_cluster_from_segment(part)
        if cluster is not None:
            return cluster
    return None


def detect_condition(path: Path, df: pd.DataFrame | None = None) -> str | None:
    path_low = path.as_posix().lower()
    before_hit = (
        bool(re.search(r'(^|[^a-z])before([^a-z]|$)', path_low))
        or bool(re.search(r'(^|[^a-z])pre([^a-z]|$)', path_low))
    )
    after_hit = (
        bool(re.search(r'(^|[^a-z])after([^a-z]|$)', path_low))
        or bool(re.search(r'(^|[^a-z])post([^a-z]|$)', path_low))
    )

    if before_hit and not after_hit:
        return 'before'
    if after_hit and not before_hit:
        return 'after'

    if df is not None and 'condition' in df.columns:
        vals = set(df['condition'].astype(str).str.lower().str.strip().unique())
        if vals == {'before'}:
            return 'before'
        if vals == {'after'}:
            return 'after'

    return None


def load_tabular_file(path: Path, nrows: int | None = None) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == '.csv':
        return pd.read_csv(path, nrows=nrows)

    if suffix == '.jsonl':
        return pd.read_json(path, lines=True, nrows=nrows)

    if suffix == '.json':
        text = path.read_text(encoding='utf-8')
        obj = json.loads(text)

        if isinstance(obj, list):
            return pd.DataFrame(obj[:nrows] if nrows is not None else obj)

        if isinstance(obj, dict):
            list_of_dict_key = None
            for k, v in obj.items():
                if isinstance(v, list) and (not v or isinstance(v[0], dict)):
                    list_of_dict_key = k
                    break
            if list_of_dict_key is not None:
                records = obj[list_of_dict_key]
                return pd.DataFrame(records[:nrows] if nrows is not None else records)
            return pd.DataFrame([obj])

    raise ValueError(f'Unsupported file format: {path}')


def detect_user_id_col(df: pd.DataFrame) -> str | None:
    cols = [str(c) for c in df.columns]
    for candidate in USER_ID_CANDIDATES:
        if candidate in cols:
            return candidate
    return None


def to_float_series(df: pd.DataFrame, col: str) -> pd.Series:
    return pd.to_numeric(df[col], errors='coerce')


def maybe_compute_metrics_with_personality_match(df: pd.DataFrame) -> pd.DataFrame:
    if compute_five_factor_metrics_pm is None:
        return df

    # Fallback: если метрик нет, но есть real/sim словари — можно пересчитать
    real_candidates = ['real_ocean', 'human_ocean', 'target_ocean', 'real_flat', 'human_profile']
    sim_candidates = ['simulated_ocean', 'model_ocean', 'pred_ocean', 'simulated_flat']

    real_col = next((c for c in real_candidates if c in df.columns), None)
    sim_col = next((c for c in sim_candidates if c in df.columns), None)
    if real_col is None or sim_col is None:
        return df

    missing_any = any(metric not in df.columns for metric in ['mae_35', 'mean_similarity_traits', 'mean_similarity_facets'])
    if not missing_any:
        return df

    computed = []
    for _, row in df.iterrows():
        real_flat = row.get(real_col)
        sim_flat = row.get(sim_col)
        if not isinstance(real_flat, dict) or not isinstance(sim_flat, dict):
            computed.append(None)
            continue
        try:
            metrics = compute_five_factor_metrics_pm(real_flat, sim_flat)
        except Exception:  # noqa: BLE001
            metrics = None
        computed.append(metrics)

    computed_series = pd.Series(computed, index=df.index)
    if 'mae_35' not in df.columns:
        df['mae_35'] = computed_series.apply(lambda x: x.get('mae_35') if isinstance(x, dict) else np.nan)
    if 'mean_similarity_traits' not in df.columns:
        df['mean_similarity_traits'] = computed_series.apply(
            lambda x: x.get('mean_similarity_traits') if isinstance(x, dict) else np.nan
        )
    if 'mean_similarity_facets' not in df.columns:
        df['mean_similarity_facets'] = computed_series.apply(
            lambda x: x.get('mean_similarity_facets') if isinstance(x, dict) else np.nan
        )

    return df


In [24]:
# --- Discovery / selection ---
def discover_candidate_files(results_root: Path) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    supported = {'.csv', '.json', '.jsonl'}

    for path in tqdm(sorted(results_root.rglob('*')), desc='Discovering files'):
        if not path.is_file() or path.suffix.lower() not in supported:
            continue

        condition_guess = detect_condition(path)
        if condition_guess is None:
            continue

        rel = path.relative_to(results_root)
        model = rel.parts[0] if len(rel.parts) >= 2 else 'unknown'
        cluster = detect_cluster(path)

        head_df = pd.DataFrame()
        load_error = None
        try:
            head_df = load_tabular_file(path, nrows=5)
        except Exception as exc:  # noqa: BLE001
            load_error = repr(exc)

        cols = [str(c) for c in head_df.columns]
        metric_cols_found = sorted({
            c for c in cols for options in METRIC_COLUMN_CANDIDATES.values() if c in options
        })
        has_user_id = detect_user_id_col(head_df) is not None
        has_answer_cols = any(re.fullmatch(r'i\d+', c.lower()) for c in cols)

        score = 0
        if metric_cols_found:
            score += 100
        if 'participants' in path.name.lower():
            score += 50
        if has_answer_cols:
            score += 20
        if has_user_id:
            score += 10
        if 'result_log' in path.as_posix().lower():
            score -= 100
        if head_df.shape[0] <= 1:
            score -= 20

        records.append(
            {
                'model': model,
                'cluster': cluster,
                'condition': condition_guess,
                'path': str(path),
                'suffix': path.suffix.lower(),
                'file_name': path.name,
                'sample_rows': int(head_df.shape[0]),
                'sample_cols': int(head_df.shape[1]),
                'columns_sample': ', '.join(cols[:20]),
                'has_user_id': bool(has_user_id),
                'has_metric_cols': bool(len(metric_cols_found) > 0),
                'metric_cols_found': ', '.join(metric_cols_found),
                'has_answer_cols': bool(has_answer_cols),
                'score': score,
                'mtime': path.stat().st_mtime,
                'load_error': load_error,
            }
        )

    if not records:
        return pd.DataFrame(columns=['model', 'cluster', 'condition', 'path'])

    return pd.DataFrame(records)


def select_best_files(candidates_df: pd.DataFrame) -> pd.DataFrame:
    if candidates_df.empty:
        return candidates_df

    eligible = candidates_df[candidates_df['cluster'].notna()].copy()
    eligible = eligible.sort_values(
        ['model', 'cluster', 'condition', 'score', 'mtime', 'path'],
        ascending=[True, True, True, False, False, True],
    )

    best = eligible.groupby(['model', 'cluster', 'condition'], as_index=False).head(1).copy()
    return best


candidates_df = discover_candidate_files(RESULTS_ROOT)
print('Найдено candidate before/after файлов:', len(candidates_df))

display_cols = [
    'model', 'cluster', 'condition', 'file_name', 'suffix',
    'sample_rows', 'sample_cols', 'has_user_id', 'has_metric_cols',
    'metric_cols_found', 'has_answer_cols', 'score', 'path', 'load_error'
]
display(candidates_df.sort_values(['model', 'cluster', 'condition', 'score'], ascending=[True, True, True, False])[display_cols])

selected_df = select_best_files(candidates_df)
print()
print('Выбрано per-user файлов для анализа:', len(selected_df))
display(selected_df.sort_values(['model', 'cluster', 'condition'])[display_cols])

# Проверка, что на каждый model×cluster есть both before+after
combo_check = selected_df.groupby(['model', 'cluster'])['condition'].agg(lambda s: sorted(set(s))).reset_index()
combo_check['has_before'] = combo_check['condition'].apply(lambda x: 'before' in x)
combo_check['has_after'] = combo_check['condition'].apply(lambda x: 'after' in x)
combo_check['ok_pair'] = combo_check['has_before'] & combo_check['has_after']

print()
print('Проверка наличия before/after на model×cluster:')
display(combo_check)

missing_pair_df = combo_check[~combo_check['ok_pair']].copy()
if not missing_pair_df.empty:
    print('WARNING: найдены model×cluster без полной пары before/after')
    display(missing_pair_df)


Discovering files:   0%|          | 0/246 [00:00<?, ?it/s]

Найдено candidate before/after файлов: 84


,model,cluster,condition,file_name,suffix,sample_rows,sample_cols,has_user_id,has_metric_cols,metric_cols_found,has_answer_cols,score,path,load_error
1,gigchat3,cluster_0,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
0,gigchat3,cluster_0,after,after_optimization_test_answers.csv,.csv,5,121,True,False,,True,30,D:\programming\GitHub\LLM-PersonaBench\results...,None
3,gigchat3,cluster_0,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
2,gigchat3,cluster_0,before,before_optimization_test_answers.csv,.csv,5,121,True,False,,True,30,D:\programming\GitHub\LLM-PersonaBench\results...,None
5,gigchat3,cluster_1,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,qwen3,cluster_2,before,before_optimization_test_answers.csv,.csv,5,121,True,False,,True,30,D:\programming\GitHub\LLM-PersonaBench\results...,None
81,qwen3,cluster_3,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
80,qwen3,cluster_3,after,after_optimization_test_answers.csv,.csv,5,121,True,False,,True,30,D:\programming\GitHub\LLM-PersonaBench\results...,None
83,qwen3,cluster_3,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None



Выбрано per-user файлов для анализа: 40


,model,cluster,condition,file_name,suffix,sample_rows,sample_cols,has_user_id,has_metric_cols,metric_cols_found,has_answer_cols,score,path,load_error
1,gigchat3,cluster_0,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
3,gigchat3,cluster_0,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
5,gigchat3,cluster_1,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
7,gigchat3,cluster_1,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
9,gigchat3,cluster_2,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
11,gigchat3,cluster_2,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
13,gigchat3,cluster_3,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
15,gigchat3,cluster_3,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
17,gpt4_mini,cluster_0,after,after_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None
19,gpt4_mini,cluster_0,before,before_optimization_test_participants.jsonl,.jsonl,5,21,True,True,"mae_35, mean_similarity_facets, mean_similarit...",False,160,D:\programming\GitHub\LLM-PersonaBench\results...,None



Проверка наличия before/after на model×cluster:


,model,cluster,condition,has_before,has_after,ok_pair
0,gigchat3,cluster_0,"[after, before]",True,True,True
1,gigchat3,cluster_1,"[after, before]",True,True,True
2,gigchat3,cluster_2,"[after, before]",True,True,True
3,gigchat3,cluster_3,"[after, before]",True,True,True
4,gpt4_mini,cluster_0,"[after, before]",True,True,True
5,gpt4_mini,cluster_1,"[after, before]",True,True,True
6,gpt4_mini,cluster_2,"[after, before]",True,True,True
7,gpt4_mini,cluster_3,"[after, before]",True,True,True
8,gpt4_nano,cluster_0,"[after, before]",True,True,True
9,gpt4_nano,cluster_1,"[after, before]",True,True,True


In [25]:
# --- Load selected files into unified long format ---
def build_long_metrics_df(selected_df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    warnings_list: list[str] = []
    frames: list[pd.DataFrame] = []

    for row in tqdm(selected_df.itertuples(index=False), total=len(selected_df), desc='Loading selected files'):
        path = Path(row.path)
        try:
            df = load_tabular_file(path, nrows=None)
        except Exception as exc:  # noqa: BLE001
            warnings_list.append(f'Failed to load {path}: {exc}')
            continue

        df = maybe_compute_metrics_with_personality_match(df)

        user_col = detect_user_id_col(df)
        if user_col is None:
            warnings_list.append(f'No user id column in {path}')
            continue

        out = pd.DataFrame(
            {
                'model': row.model,
                'cluster': row.cluster,
                'condition': row.condition,
                'user_id': df[user_col].astype(str),
                'source_file': str(path),
            }
        )

        metric_missing_for_file = []
        for metric, col_candidates in METRIC_COLUMN_CANDIDATES.items():
            chosen_col = next((c for c in col_candidates if c in df.columns), None)
            if chosen_col is not None:
                out[metric] = to_float_series(df, chosen_col)
            elif metric == 'answer_similarity' and 'avg_diff' in df.columns:
                # fallback, если есть avg_diff
                out[metric] = 1.0 - to_float_series(df, 'avg_diff') / 4.0
            else:
                out[metric] = np.nan
                metric_missing_for_file.append(metric)

        if metric_missing_for_file:
            warnings_list.append(
                f"Missing metrics in {path.name}: {', '.join(metric_missing_for_file)}"
            )

        frames.append(out)

    if not frames:
        return pd.DataFrame(columns=['model', 'cluster', 'condition', 'user_id', *METRICS]), warnings_list

    long_df = pd.concat(frames, ignore_index=True)

    # На случай дублей user_id внутри одного source condition: агрегируем mean
    long_df = (
        long_df.groupby(['model', 'cluster', 'condition', 'user_id'], as_index=False)[METRICS]
        .mean()
    )

    return long_df, warnings_list


long_df, load_warnings = build_long_metrics_df(selected_df)
print('Размер long_df:', long_df.shape)
display(long_df.head())

users_before_pairing_df = (
    long_df.groupby(['model', 'cluster', 'condition'], as_index=False)['user_id']
    .nunique()
    .rename(columns={'user_id': 'n_users'})
    .sort_values(['model', 'cluster', 'condition'])
)

print()
print('Количество пользователей до pairing (model×cluster×condition):')
display(users_before_pairing_df)

if load_warnings:
    print()
    print('WARNING: проблемы при загрузке/метриках:')
    display(pd.DataFrame({'warning': load_warnings}))


Loading selected files:   0%|          | 0/40 [00:00<?, ?it/s]

Размер long_df: (1600, 8)


,model,cluster,condition,user_id,answer_similarity,mae_35,trait_similarity,facet_similarity
0,gigchat3,cluster_0,after,529,0.695833,24.611642,0.776161,0.750171
1,gigchat3,cluster_0,after,540,0.689583,27.945727,0.797689,0.707685
2,gigchat3,cluster_0,after,542,0.718750,28.443267,0.729411,0.713260
3,gigchat3,cluster_0,after,547,0.787500,20.677137,0.803363,0.791540
4,gigchat3,cluster_0,after,570,0.697917,23.371709,0.810774,0.758868



Количество пользователей до pairing (model×cluster×condition):


,model,cluster,condition,n_users
0,gigchat3,cluster_0,after,40
1,gigchat3,cluster_0,before,40
2,gigchat3,cluster_1,after,40
3,gigchat3,cluster_1,before,40
4,gigchat3,cluster_2,after,40
5,gigchat3,cluster_2,before,40
6,gigchat3,cluster_3,after,40
7,gigchat3,cluster_3,before,40
8,gpt4_mini,cluster_0,after,40
9,gpt4_mini,cluster_0,before,40


In [26]:
# --- Build paired wide dataframe ---
def build_pairing(long_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    diagnostics = []
    paired_frames = []

    for (model, cluster), group in long_df.groupby(['model', 'cluster']):
        before = group[group['condition'] == 'before'].set_index('user_id')
        after = group[group['condition'] == 'after'].set_index('user_id')

        b_users = set(before.index)
        a_users = set(after.index)
        paired_users = sorted(b_users & a_users)

        diagnostics.append(
            {
                'model': model,
                'cluster': cluster,
                'n_before_users': len(b_users),
                'n_after_users': len(a_users),
                'n_paired_users': len(paired_users),
                'n_excluded_before_only': len(b_users - a_users),
                'n_excluded_after_only': len(a_users - b_users),
            }
        )

        if paired_users:
            merged = before.loc[paired_users, METRICS].add_suffix('_before').join(
                after.loc[paired_users, METRICS].add_suffix('_after'),
                how='inner',
            )
            merged = merged.reset_index().rename(columns={'index': 'user_id'})
            merged.insert(0, 'cluster', cluster)
            merged.insert(0, 'model', model)
            paired_frames.append(merged)

    pairing_diag_df = pd.DataFrame(diagnostics).sort_values(['model', 'cluster']).reset_index(drop=True)

    if paired_frames:
        wide_df = pd.concat(paired_frames, ignore_index=True)
    else:
        wide_cols = ['model', 'cluster', 'user_id']
        for metric in METRICS:
            wide_cols.extend([f'{metric}_before', f'{metric}_after'])
        wide_df = pd.DataFrame(columns=wide_cols)

    return wide_df, pairing_diag_df


wide_df, pairing_diag_df = build_pairing(long_df)

print('Размер wide_df (paired users):', wide_df.shape)
print()
print('Диагностика pairing по model×cluster:')
display(pairing_diag_df)

unpaired_df = pairing_diag_df[
    (pairing_diag_df['n_excluded_before_only'] > 0)
    | (pairing_diag_df['n_excluded_after_only'] > 0)
].copy()

if not unpaired_df.empty:
    print('WARNING: есть непарные записи, исключённые из paired анализа')
    display(unpaired_df)


Размер wide_df (paired users): (800, 11)

Диагностика pairing по model×cluster:


,model,cluster,n_before_users,n_after_users,n_paired_users,n_excluded_before_only,n_excluded_after_only
0,gigchat3,cluster_0,40,40,40,0,0
1,gigchat3,cluster_1,40,40,40,0,0
2,gigchat3,cluster_2,40,40,40,0,0
3,gigchat3,cluster_3,40,40,40,0,0
4,gpt4_mini,cluster_0,40,40,40,0,0
5,gpt4_mini,cluster_1,40,40,40,0,0
6,gpt4_mini,cluster_2,40,40,40,0,0
7,gpt4_mini,cluster_3,40,40,40,0,0
8,gpt4_nano,cluster_0,40,40,40,0,0
9,gpt4_nano,cluster_1,40,40,40,0,0


In [27]:
# --- Paired bootstrap ---
def paired_bootstrap_ci(
    delta: np.ndarray | pd.Series,
    n_resamples: int = 10_000,
    confidence_level: float = 0.95,
    seed: int = 42,
) -> dict[str, Any]:
    arr = np.asarray(delta, dtype=float)
    arr = arr[np.isfinite(arr)]

    out: dict[str, Any] = {
        'n_users': int(arr.size),
        'mean_delta': np.nan,
        'ci_low': np.nan,
        'ci_high': np.nan,
        'significant': False,
        'bootstrap_method': None,
        'warning': None,
    }

    if arr.size == 0:
        out['warning'] = 'No finite deltas'
        return out

    out['mean_delta'] = float(arr.mean())

    if arr.size < 2:
        out['warning'] = 'Need at least 2 paired users for bootstrap CI'
        return out

    try:
        res = bootstrap(
            (arr,),
            statistic=np.mean,
            n_resamples=n_resamples,
            confidence_level=confidence_level,
            method='BCa',
            random_state=seed,
            vectorized=False,
        )
        ci_low = float(res.confidence_interval.low)
        ci_high = float(res.confidence_interval.high)

        if not np.isfinite(ci_low) or not np.isfinite(ci_high):
            raise ValueError('BCa returned non-finite CI bounds')

        out['ci_low'] = ci_low
        out['ci_high'] = ci_high
        out['bootstrap_method'] = 'BCa'
    except Exception as exc:  # noqa: BLE001
        rng = np.random.default_rng(seed)
        samples = rng.choice(arr, size=(n_resamples, arr.size), replace=True)
        boot_means = samples.mean(axis=1)
        alpha = (1.0 - confidence_level) / 2.0
        ci_low, ci_high = np.percentile(boot_means, [100 * alpha, 100 * (1 - alpha)])

        out['ci_low'] = float(ci_low)
        out['ci_high'] = float(ci_high)
        out['bootstrap_method'] = 'percentile_fallback'
        out['warning'] = f'BCa failed, fallback used: {exc}'

    out['significant'] = bool(out['ci_low'] > 0 or out['ci_high'] < 0)
    return out


def summarize_significance(
    wide_df: pd.DataFrame,
    n_resamples: int = 10_000,
    seed: int = 42,
) -> tuple[pd.DataFrame, list[str]]:
    rows = []
    warn = []

    group_cols = ['model', 'cluster']

    for (model, cluster), chunk in wide_df.groupby(group_cols):
        for metric in METRICS:
            before_col = f'{metric}_before'
            after_col = f'{metric}_after'

            before_vals = pd.to_numeric(chunk[before_col], errors='coerce').to_numpy(dtype=float)
            after_vals = pd.to_numeric(chunk[after_col], errors='coerce').to_numpy(dtype=float)

            # Главное исправление:
            # оставляем только пользователей, у которых есть и before, и after
            finite_mask = np.isfinite(before_vals) & np.isfinite(after_vals)

            before_complete = before_vals[finite_mask]
            after_complete = after_vals[finite_mask]
            delta = after_complete - before_complete

            n_total_paired_users = len(chunk)
            n_valid_metric_pairs = int(finite_mask.sum())
            n_missing_metric_pairs = int(n_total_paired_users - n_valid_metric_pairs)

            if n_missing_metric_pairs > 0:
                warn.append(
                    f"{model}/{cluster}/{metric}: "
                    f"{n_missing_metric_pairs} paired users excluded because "
                    f"before or after metric is missing"
                )

            bs = paired_bootstrap_ci(
                delta=delta,
                n_resamples=n_resamples,
                confidence_level=0.95,
                seed=seed,
            )

            direction = METRIC_DIRECTIONS[metric]

            if direction == 'higher_is_better':
                improvement_significant = bool(np.isfinite(bs['ci_low']) and bs['ci_low'] > 0)
            else:
                improvement_significant = bool(np.isfinite(bs['ci_high']) and bs['ci_high'] < 0)

            if bs.get('warning'):
                warn.append(f"{model}/{cluster}/{metric}: {bs['warning']}")

            rows.append(
                {
                    'model': model,
                    'cluster': cluster,
                    'metric': metric,

                    # Полезно хранить оба числа:
                    # всего paired users и сколько реально вошло в расчет метрики
                    'n_paired_users_total': n_total_paired_users,
                    'n_users': n_valid_metric_pairs,
                    'n_missing_metric_pairs': n_missing_metric_pairs,

                    # Важно: mean_before и mean_after считаются на тех же пользователях,
                    # что и delta/bootstrap
                    'mean_before': float(before_complete.mean()) if n_valid_metric_pairs > 0 else np.nan,
                    'mean_after': float(after_complete.mean()) if n_valid_metric_pairs > 0 else np.nan,
                    'mean_delta': bs['mean_delta'],

                    'ci_low': bs['ci_low'],
                    'ci_high': bs['ci_high'],
                    'significant': bool(bs['significant']),
                    'improvement_significant': bool(improvement_significant),
                    'direction': direction,
                    'bootstrap_method': bs['bootstrap_method'],
                }
            )

    summary_df = (
        pd.DataFrame(rows)
        .sort_values(['model', 'cluster', 'metric'])
        .reset_index(drop=True)
    )

    return summary_df, warn


def format_delta_ci(row: pd.Series) -> str:
    metric = row['metric']
    decimals = 3 if metric == 'mae_35' else 4

    if not (np.isfinite(row['mean_delta']) and np.isfinite(row['ci_low']) and np.isfinite(row['ci_high'])):
        return 'n/a'

    star = ' *' if bool(row.get('improvement_significant', False)) else ''
    return (
        f"{row['mean_delta']:+.{decimals}f} "
        f"[{row['ci_low']:+.{decimals}f}, {row['ci_high']:+.{decimals}f}]"
        f"{star}"
    )


summary_df, bootstrap_warnings = summarize_significance(wide_df, n_resamples=10_000, seed=42)
summary_df['formatted_delta_ci'] = summary_df.apply(format_delta_ci, axis=1)

print('Размер summary_df:', summary_df.shape)
display(summary_df)

if bootstrap_warnings:
    print('WARNING: bootstrap warnings')
    display(pd.DataFrame({'warning': bootstrap_warnings}))


Размер summary_df: (80, 16)


,model,cluster,metric,n_paired_users_total,n_users,n_missing_metric_pairs,mean_before,mean_after,mean_delta,ci_low,ci_high,significant,improvement_significant,direction,bootstrap_method,formatted_delta_ci
0,gigchat3,cluster_0,answer_similarity,40,38,2,0.664912,0.712901,0.047988,0.034793,0.061298,True,True,higher_is_better,BCa,"+0.0480 [+0.0348, +0.0613] *"
1,gigchat3,cluster_0,facet_similarity,40,37,3,0.747101,0.730418,-0.016683,-0.028427,-0.005436,True,False,higher_is_better,BCa,"-0.0167 [-0.0284, -0.0054]"
2,gigchat3,cluster_0,mae_35,40,37,3,24.710970,26.215783,1.504813,0.416117,2.643351,True,False,lower_is_better,BCa,"+1.505 [+0.416, +2.643]"
3,gigchat3,cluster_0,trait_similarity,40,37,3,0.787629,0.782389,-0.005240,-0.021437,0.012676,False,False,higher_is_better,BCa,"-0.0052 [-0.0214, +0.0127]"
4,gigchat3,cluster_1,answer_similarity,40,35,5,0.675417,0.689643,0.014226,0.003411,0.025952,True,True,higher_is_better,BCa,"+0.0142 [+0.0034, +0.0260] *"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,qwen3,cluster_2,trait_similarity,40,40,0,0.761994,0.734682,-0.027312,-0.049255,-0.006252,True,False,higher_is_better,BCa,"-0.0273 [-0.0493, -0.0063]"
76,qwen3,cluster_3,answer_similarity,40,40,0,0.642552,0.642083,-0.000469,-0.008177,0.007552,False,False,higher_is_better,BCa,"-0.0005 [-0.0082, +0.0076]"
77,qwen3,cluster_3,facet_similarity,40,40,0,0.690351,0.685673,-0.004677,-0.014554,0.003992,False,False,higher_is_better,BCa,"-0.0047 [-0.0146, +0.0040]"
78,qwen3,cluster_3,mae_35,40,40,0,30.830760,31.239484,0.408724,-0.508788,1.780736,False,False,lower_is_better,BCa,"+0.409 [-0.509, +1.781]"


,warning
0,gigchat3/cluster_0/answer_similarity: 2 paired...
1,gigchat3/cluster_0/mae_35: 3 paired users excl...
2,gigchat3/cluster_0/trait_similarity: 3 paired ...
3,gigchat3/cluster_0/facet_similarity: 3 paired ...
4,gigchat3/cluster_1/answer_similarity: 5 paired...
5,gigchat3/cluster_1/mae_35: 5 paired users excl...
6,gigchat3/cluster_1/trait_similarity: 5 paired ...
7,gigchat3/cluster_1/facet_similarity: 5 paired ...
8,gigchat3/cluster_2/answer_similarity: 1 paired...
9,gigchat3/cluster_2/mae_35: 1 paired users excl...


In [28]:
# --- Build tables A/B/C + paired deltas ---
def build_outputs(
    wide_df: pd.DataFrame,
    summary_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    summary_long = summary_df[
        [
            'model',
            'cluster',
            'metric',
            'n_users',
            'mean_before',
            'mean_after',
            'mean_delta',
            'ci_low',
            'ci_high',
            'significant',
            'improvement_significant',
            'direction',
            'bootstrap_method',
            'formatted_delta_ci',
        ]
    ].copy() if not summary_df.empty else pd.DataFrame(
        columns=[
            'model', 'cluster', 'metric', 'n_users', 'mean_before', 'mean_after',
            'mean_delta', 'ci_low', 'ci_high', 'significant', 'improvement_significant',
            'direction', 'bootstrap_method', 'formatted_delta_ci'
        ]
    )

    pivot_df = summary_long.pivot_table(
        index=['model', 'cluster'],
        columns='metric',
        values='formatted_delta_ci',
        aggfunc='first',
    ).reset_index() if not summary_long.empty else pd.DataFrame(columns=['model', 'cluster', *METRICS])

    answer_only_df = summary_long[summary_long['metric'] == 'answer_similarity'][
        ['model', 'cluster', 'n_users', 'mean_delta', 'ci_low', 'ci_high', 'improvement_significant']
    ].reset_index(drop=True)

    paired_user_deltas_df = wide_df[['model', 'cluster', 'user_id']].copy() if not wide_df.empty else pd.DataFrame(
        columns=['model', 'cluster', 'user_id']
    )
    for metric in METRICS:
        if not wide_df.empty:
            paired_user_deltas_df[f'{metric}_delta'] = wide_df[f'{metric}_after'] - wide_df[f'{metric}_before']
        else:
            paired_user_deltas_df[f'{metric}_delta'] = []

    return summary_long, pivot_df, answer_only_df, paired_user_deltas_df


summary_long, pivot_df, answer_only_df, paired_user_deltas_df = build_outputs(wide_df, summary_df)

print('A) long summary')
display(summary_long)

print('B) pivot by model-cluster')
display(pivot_df)

print('C) answer_similarity only')
display(answer_only_df)


A) long summary


,model,cluster,metric,n_users,mean_before,mean_after,mean_delta,ci_low,ci_high,significant,improvement_significant,direction,bootstrap_method,formatted_delta_ci
0,gigchat3,cluster_0,answer_similarity,38,0.664912,0.712901,0.047988,0.034793,0.061298,True,True,higher_is_better,BCa,"+0.0480 [+0.0348, +0.0613] *"
1,gigchat3,cluster_0,facet_similarity,37,0.747101,0.730418,-0.016683,-0.028427,-0.005436,True,False,higher_is_better,BCa,"-0.0167 [-0.0284, -0.0054]"
2,gigchat3,cluster_0,mae_35,37,24.710970,26.215783,1.504813,0.416117,2.643351,True,False,lower_is_better,BCa,"+1.505 [+0.416, +2.643]"
3,gigchat3,cluster_0,trait_similarity,37,0.787629,0.782389,-0.005240,-0.021437,0.012676,False,False,higher_is_better,BCa,"-0.0052 [-0.0214, +0.0127]"
4,gigchat3,cluster_1,answer_similarity,35,0.675417,0.689643,0.014226,0.003411,0.025952,True,True,higher_is_better,BCa,"+0.0142 [+0.0034, +0.0260] *"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,qwen3,cluster_2,trait_similarity,40,0.761994,0.734682,-0.027312,-0.049255,-0.006252,True,False,higher_is_better,BCa,"-0.0273 [-0.0493, -0.0063]"
76,qwen3,cluster_3,answer_similarity,40,0.642552,0.642083,-0.000469,-0.008177,0.007552,False,False,higher_is_better,BCa,"-0.0005 [-0.0082, +0.0076]"
77,qwen3,cluster_3,facet_similarity,40,0.690351,0.685673,-0.004677,-0.014554,0.003992,False,False,higher_is_better,BCa,"-0.0047 [-0.0146, +0.0040]"
78,qwen3,cluster_3,mae_35,40,30.830760,31.239484,0.408724,-0.508788,1.780736,False,False,lower_is_better,BCa,"+0.409 [-0.509, +1.781]"


B) pivot by model-cluster


metric,model,cluster,answer_similarity,facet_similarity,mae_35,trait_similarity
0,gigchat3,cluster_0,"+0.0480 [+0.0348, +0.0613] *","-0.0167 [-0.0284, -0.0054]","+1.505 [+0.416, +2.643]","-0.0052 [-0.0214, +0.0127]"
1,gigchat3,cluster_1,"+0.0142 [+0.0034, +0.0260] *","-0.0076 [-0.0160, +0.0001]","+0.435 [-0.305, +1.234]","+0.0153 [+0.0014, +0.0279] *"
2,gigchat3,cluster_2,"-0.0021 [-0.0082, +0.0058]","-0.0029 [-0.0107, +0.0054]","+0.390 [-0.467, +1.183]","-0.0101 [-0.0246, +0.0077]"
3,gigchat3,cluster_3,"+0.0350 [+0.0131, +0.0609] *","+0.0163 [-0.0006, +0.0351]","-1.323 [-3.105, +0.275]","-0.0054 [-0.0272, +0.0163]"
4,gpt4_mini,cluster_0,"+0.0030 [-0.0021, +0.0074]","-0.0019 [-0.0106, +0.0033]","+0.322 [-0.202, +1.194]","-0.0114 [-0.0224, -0.0022]"
5,gpt4_mini,cluster_1,"+0.0017 [-0.0010, +0.0046]","-0.0008 [-0.0036, +0.0023]","+0.055 [-0.275, +0.339]","+0.0009 [-0.0040, +0.0063]"
6,gpt4_mini,cluster_2,"+0.0228 [+0.0155, +0.0303] *","+0.0273 [+0.0136, +0.0418] *","-2.151 [-3.769, -0.619] *","-0.0135 [-0.0452, +0.0204]"
7,gpt4_mini,cluster_3,"+0.0047 [-0.0004, +0.0101]","-0.0105 [-0.0167, -0.0046]","+1.179 [+0.552, +1.771]","-0.0193 [-0.0311, -0.0052]"
8,gpt4_nano,cluster_0,"+0.0002 [-0.0030, +0.0033]","-0.0004 [-0.0049, +0.0046]","+0.009 [-0.502, +0.474]","+0.0017 [-0.0058, +0.0095]"
9,gpt4_nano,cluster_1,"+0.0048 [+0.0017, +0.0082] *","+0.0010 [-0.0032, +0.0058]","-0.218 [-0.661, +0.202]","+0.0090 [+0.0006, +0.0169] *"


C) answer_similarity only


,model,cluster,n_users,mean_delta,ci_low,ci_high,improvement_significant
0,gigchat3,cluster_0,38,0.047988,0.034793,0.061298,True
1,gigchat3,cluster_1,35,0.014226,0.003411,0.025952,True
2,gigchat3,cluster_2,39,-0.002137,-0.008226,0.005823,False
3,gigchat3,cluster_3,40,0.035000,0.013149,0.060937,True
4,gpt4_mini,cluster_0,40,0.002969,-0.002083,0.007448,False
5,gpt4_mini,cluster_1,40,0.001667,-0.001042,0.004583,False
6,gpt4_mini,cluster_2,40,0.022813,0.015469,0.030260,True
7,gpt4_mini,cluster_3,40,0.004687,-0.000417,0.010104,False
8,gpt4_nano,cluster_0,40,0.000156,-0.002969,0.003333,False
9,gpt4_nano,cluster_1,40,0.004844,0.001667,0.008229,True


In [29]:
# --- Save outputs ---
summary_csv = OUTPUT_DIR / 'stat_significance_summary.csv'
summary_xlsx = OUTPUT_DIR / 'stat_significance_summary.xlsx'
pivot_xlsx = OUTPUT_DIR / 'stat_significance_pivot.xlsx'
paired_csv = OUTPUT_DIR / 'paired_user_deltas.csv'

summary_long.to_csv(summary_csv, index=False)
paired_user_deltas_df.to_csv(paired_csv, index=False)

excel_engine = None
if importlib.util.find_spec('openpyxl') is not None:
    excel_engine = 'openpyxl'
elif importlib.util.find_spec('xlsxwriter') is not None:
    excel_engine = 'xlsxwriter'

if excel_engine is not None:
    with pd.ExcelWriter(summary_xlsx, engine=excel_engine) as writer:
        summary_long.to_excel(writer, sheet_name='summary_long', index=False)
        answer_only_df.to_excel(writer, sheet_name='answer_similarity', index=False)
        pairing_diag_df.to_excel(writer, sheet_name='pairing_diagnostics', index=False)

    with pd.ExcelWriter(pivot_xlsx, engine=excel_engine) as writer:
        pivot_df.to_excel(writer, sheet_name='pivot_model_cluster', index=False)
        answer_only_df.to_excel(writer, sheet_name='answer_similarity', index=False)

print('Saved:', summary_csv)
print('Saved:', paired_csv)
if excel_engine is not None:
    print('Saved:', summary_xlsx)
    print('Saved:', pivot_xlsx)
else:
    print('WARNING: openpyxl/xlsxwriter не найден; xlsx-файлы не сохранены в этой среде.')


Saved: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2\stat_significance\stat_significance_summary.csv
Saved: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2\stat_significance\paired_user_deltas.csv
Saved: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2\stat_significance\stat_significance_summary.xlsx
Saved: D:\programming\GitHub\LLM-PersonaBench\results_experiments\evoprompt_iter2\stat_significance\stat_significance_pivot.xlsx


In [30]:
# --- Consolidated diagnostics ---
all_warnings = []

if not missing_pair_df.empty:
    all_warnings.append('Есть model×cluster без полной пары before/after (см. таблицу выше).')

if load_warnings:
    all_warnings.extend(load_warnings)

if bootstrap_warnings:
    all_warnings.extend(bootstrap_warnings)

if not PERSONALITY_MATCH_AVAILABLE:
    all_warnings.append(
        'Не удалось импортировать src.utils.personality_match (зависимости окружения). '        'Для текущих participants-файлов это не блокирует расчёт, т.к. метрики уже сохранены per-user.'
    )

print('Итоговые размеры:')
print(' - candidates_df:', candidates_df.shape)
print(' - selected_df:', selected_df.shape)
print(' - long_df:', long_df.shape)
print(' - wide_df (paired):', wide_df.shape)
print(' - summary_long:', summary_long.shape)
print(' - pivot_df:', pivot_df.shape)
print(' - paired_user_deltas_df:', paired_user_deltas_df.shape)

if all_warnings:
    print()
    print('Warnings:')
    display(pd.DataFrame({'warning': all_warnings}))
else:
    print()
    print('Warnings: none')


Итоговые размеры:
 - candidates_df: (84, 16)
 - selected_df: (40, 16)
 - long_df: (1600, 8)
 - wide_df (paired): (800, 11)
 - summary_long: (80, 14)
 - pivot_df: (20, 6)
 - paired_user_deltas_df: (800, 7)

Warnings:


,warning
0,gigchat3/cluster_0/answer_similarity: 2 paired...
1,gigchat3/cluster_0/mae_35: 3 paired users excl...
2,gigchat3/cluster_0/trait_similarity: 3 paired ...
3,gigchat3/cluster_0/facet_similarity: 3 paired ...
4,gigchat3/cluster_1/answer_similarity: 5 paired...
5,gigchat3/cluster_1/mae_35: 5 paired users excl...
6,gigchat3/cluster_1/trait_similarity: 5 paired ...
7,gigchat3/cluster_1/facet_similarity: 5 paired ...
8,gigchat3/cluster_2/answer_similarity: 1 paired...
9,gigchat3/cluster_2/mae_35: 1 paired users excl...
